# Week 9 Lab — Probability, Bayes' Theorem, and Medical Testing
## SCIE1500 Science and Quantitative Reasoning — Semester 2, 2026

**Lab format:** Work in your groups. Run all code cells in order, fill in the written prompts.

---
**This week:** We use probability and Bayes' theorem to evaluate the real-world performance of diagnostic tests — a critical skill in public health and clinical science.

---
## 📋 Scientific Scenario

During a COVID-19 wave, health authorities use a rapid antigen test with the following properties:

- **Sensitivity** (true positive rate): 85% — if you have COVID, the test detects it 85% of the time.
- **Specificity** (true negative rate): 95% — if you don't have COVID, the test correctly gives a negative 95% of the time.
- **Community prevalence**: 2% of the population currently has COVID.

**Question:** If a randomly chosen person tests positive, what is the probability they actually have COVID?

---
## 🎯 Group Task & Learning Objectives

| Part | Topic | Time |
|------|-------|------|
| A | Bayes' theorem: algebra and intuition | ~25 min |
| B | Python calculation and verification | ~20 min |
| C | Sensitivity analysis: how prevalence changes everything | ~15 min |

By the end you should be able to: apply Bayes' theorem to diagnostic testing; interpret sensitivity, specificity, and positive predictive value (PPV); and explain why prevalence drives PPV.

In [ ]:
# Run this cell first — loads libraries for today's lab
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Test parameters
sensitivity  = 0.85   # P(T+ | Disease)
specificity  = 0.95   # P(T- | No Disease)
prevalence   = 0.02   # P(Disease)

false_pos_rate = 1 - specificity   # P(T+ | No Disease)

print("Test characteristics:")
print(f"  Sensitivity:       {sensitivity:.0%}")
print(f"  Specificity:       {specificity:.0%}")
print(f"  False positive:    {false_pos_rate:.0%}")
print(f"  Prevalence:        {prevalence:.0%}")

---
## Part A: Bayes' Theorem (~25 min)

We want $P(D \mid T^+)$ — the probability of disease given a positive test (the **Positive Predictive Value**, PPV).

**Step 1** — Law of Total Probability for $P(T^+)$:
$$P(T^+) = P(T^+ \mid D) \cdot P(D) + P(T^+ \mid \bar{D}) \cdot P(\bar{D})$$
$$= 0.85 \times 0.02 + 0.05 \times 0.98 = 0.017 + 0.049 = 0.066$$

**Step 2** — Bayes' theorem:
$$P(D \mid T^+) = \frac{P(T^+ \mid D) \cdot P(D)}{P(T^+)} = \frac{0.017}{0.066} \approx 0.257$$

✏️ **A.1** In plain English: if 10,000 people are tested, how many true positives and false positives do you expect? Fill in the table:

| Group | Count | Test result |
|-------|-------|-------------|
| Have COVID (2% of 10,000) | 200 | T+ : ?, T- : ? |
| No COVID (98% of 10,000) | 9,800 | T+ : ?, T- : ? |
| **Total T+** | ? | |

```
ANSWERS:
...
```

In [ ]:
# A.1 — Verify Bayes' theorem
P_D    = prevalence
P_Dbar = 1 - P_D

# Joint probabilities
P_Tplus_and_D    = sensitivity * P_D
P_Tplus_and_Dbar = false_pos_rate * P_Dbar

P_Tplus = P_Tplus_and_D + P_Tplus_and_Dbar

PPV = P_Tplus_and_D / P_Tplus

print("Calculation:")
print(f"  P(T+ ∩ D)    = {sensitivity}×{P_D}   = {P_Tplus_and_D:.4f}")
print(f"  P(T+ ∩ D̄)   = {false_pos_rate}×{P_Dbar:.2f} = {P_Tplus_and_Dbar:.4f}")
print(f"  P(T+)        = {P_Tplus:.4f}")
print(f"  PPV = P(D|T+) = {PPV:.4f} ≈ {PPV:.1%}")
print(f"\n→ Only {PPV:.1%} of positive tests are truly positive!")

In [ ]:
# A.2 — The 10,000 person table
N = 10000
n_D    = N * P_D
n_Dbar = N * P_Dbar

TP = sensitivity     * n_D       # true positives
FN = (1-sensitivity) * n_D       # false negatives
FP = false_pos_rate  * n_Dbar    # false positives
TN = specificity     * n_Dbar    # true negatives

print("Confusion matrix (10,000 people):")
print(f"{'':20s} {'T+':>8} {'T-':>8} {'Total':>8}")
print(f"{'Have COVID':20s} {TP:>8.0f} {FN:>8.0f} {n_D:>8.0f}")
print(f"{'No COVID':20s} {FP:>8.0f} {TN:>8.0f} {n_Dbar:>8.0f}")
print(f"{'Total':20s} {TP+FP:>8.0f} {FN+TN:>8.0f} {N:>8.0f}")
print(f"\nPPV = TP / (TP+FP) = {TP:.0f} / {TP+FP:.0f} = {TP/(TP+FP):.1%}")

---
## Part B: Python Verification and NPV (~20 min)

In [ ]:
# B.1 — Negative Predictive Value (NPV): P(no disease | T-)
P_Tminus = 1 - P_Tplus
P_D_given_Tminus = ((1 - sensitivity) * P_D) / P_Tminus
NPV = 1 - P_D_given_Tminus

print(f"P(T-)   = {P_Tminus:.4f}")
print(f"P(D|T-) = {P_D_given_Tminus:.4f}  (prob. you have COVID despite negative test)")
print(f"NPV     = {NPV:.4f} = {NPV:.2%}  (prob. truly disease-free given negative test)")

---
## Part C: Sensitivity Analysis (~15 min)

How does PPV change as community prevalence changes?

In [ ]:
# C.1 — PPV vs prevalence curve
prev_vals = np.linspace(0.001, 0.50, 500)

def bayes_PPV(prev, sens=sensitivity, spec=specificity):
    'Positive Predictive Value for given prevalence.'
    fpr = 1 - spec
    P_pos = sens * prev + fpr * (1 - prev)
    return sens * prev / P_pos

ppv_vals = bayes_PPV(prev_vals)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prev_vals * 100, ppv_vals * 100, "steelblue", lw=2.5)
ax.axvline(prevalence * 100, color="red", ls="--", label=f"Current prevalence ({prevalence:.0%})")
ax.axhline(PPV * 100, color="red", ls=":", alpha=0.7, label=f"Current PPV ({PPV:.1%})")
ax.fill_between(prev_vals * 100, ppv_vals * 100, alpha=0.15, color="steelblue")
ax.set_xlabel("Community Prevalence (%)")
ax.set_ylabel("Positive Predictive Value (%)")
ax.set_title(f"PPV vs Prevalence (Sensitivity={sensitivity:.0%}, Specificity={specificity:.0%})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PPV at 2% prevalence:  {bayes_PPV(0.02):.1%}")
print(f"PPV at 10% prevalence: {bayes_PPV(0.10):.1%}")
print(f"PPV at 30% prevalence: {bayes_PPV(0.30):.1%}")

✏️ **Group Discussion:**

1. Why is the PPV only ~26% even though the test is 85% sensitive? Where do the false positives come from?
2. At what prevalence does the PPV reach 50%? Use your plot to estimate, then verify with Python.
3. Should health authorities use this test for population screening at 2% prevalence? What are the consequences of a false positive for the patient?

```
DISCUSSION:
...
```

---
## ✅ Submission Exercise — Batch 3 (due Tuesday 29 September, 11:59 pm)

**Submit via LMS. Covers Weeks 7–9 (definite integrals, ODEs, probability).**

Work individually. Show all working. Write Python code where indicated.

---
### Q1 — Consumer and Producer Surplus (Week 7)

A freshwater prawn market in the Ord River region has:
- Demand: $Q_d = 400 - 2P$ (kg/week)
- Supply: $Q_s = -60 + 3P$ (kg/week)

**(a)** Find the equilibrium price $P^*$ and quantity $Q^*$ algebraically.

**(b)** Derive the inverse demand and inverse supply functions.

**(c)** Calculate CS and PS using the triangle formulas. Show all working.

**(d)** Write Python code to verify CS and PS using numerical integration (`numpy.trapz`).

```python
# Q1d — verify with integration
import numpy as np

def demand_P(Q): ...
def supply_P(Q): ...

# YOUR CODE HERE
```

---
### Q2 — Predator-Prey Dynamics (Week 8)

A quoll-rat system in the Pilbara:

$$\frac{dR}{dt} = 0.6R - 0.003RQ, \qquad \frac{dQ}{dt} = -0.2Q + 0.001RQ$$

where $R$ = rat population, $Q$ = quoll population.

**(a)** Find the coexistence equilibrium $(R^*, Q^*)$. Show your algebra.

**(b)** Identify the rat nullcline and quoll nullcline. What does each represent biologically?

**(c)** Write Python code to simulate the system for 30 years starting from $R_0 = 500$, $Q_0 = 80$ using Euler's method (dt = 0.01). Plot both populations over time.

```python
# Q2c — Euler method simulation
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Parameters
r_R = 0.6; a = 0.003; d_Q = 0.2; b = 0.001

# YOUR CODE HERE
```

**(d)** Interpret: do the populations oscillate or converge? Is the equilibrium reached?

---
### Q3 — Bayes' Theorem (Week 9)

A breast cancer screening test has sensitivity 90% and specificity 92%. The background prevalence of breast cancer in the 50–60 age group is 1.5%.

**(a)** Calculate $P(T^+)$ — the probability a randomly selected person tests positive.

**(b)** Calculate the PPV using Bayes' theorem. Show each step.

**(c)** Write Python code to plot PPV vs prevalence for prevalence ranging from 0.1% to 20%. Annotate the 1.5% prevalence point.

```python
# Q3c — PPV vs prevalence
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

sensitivity = 0.90
specificity = 0.92

# YOUR CODE HERE
```

**(d)** At what prevalence would PPV reach 50%? Solve algebraically or numerically.